<a href="https://colab.research.google.com/github/athulkkrishnann-cloud/Used-Car-Data-Preprocessing/blob/main/UsedCarData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Day12_Used_Car_Preprocessing_Dataset.csv to Day12_Used_Car_Preprocessing_Dataset.csv


In [6]:
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

df.head()

,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


In [13]:
df.shape

(320, 15)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    object 
 1   Brand               320 non-null    object 
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    object 
 7   Transmission        320 non-null    object 
 8   City                320 non-null    object 
 9   Seller_Type         320 non-null    object 
 10  Condition           320 non-null    object 
 11  Previous_Owners     320 non-null    int64  
 12  Accidents_Reported  320 non-null    int64  
 13  Service_Score       320 non-null    int64  
 14  Resale_Price_Lakh   320 non-null    float64
dtypes: float64(2), int64(6), object(7)
memory usage: 37.6+ KB

In [16]:
print("Duplicates:", df.duplicated().sum())

Duplicates: 0


In [15]:
df.select_dtypes(include="object").columns

Index(['Car_ID', 'Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type',
       'Condition'],
      dtype='object')

In [18]:
X = df.drop("Resale_Price_Lakh", axis=1)
y = df["Resale_Price_Lakh"]

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (256, 14)
Testing data: (64, 14)


In [21]:
numeric_columns = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Previous_Owners",
    "Accidents_Reported",
    "Service_Score"
]

for column in numeric_columns:
    Q1 = X_train[column].quantile(0.25)
    Q3 = X_train[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    X_train[column] = X_train[column].clip(lower, upper)
    X_test[column] = X_test[column].clip(lower, upper)

In [23]:

X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)


X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

In [27]:
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (256, 290)
Testing shape: (64, 290)


In [28]:
X_train.head()

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Car_ID_CAR0002,Car_ID_CAR0003,Car_ID_CAR0005,...,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Dealer,Seller_Type_Individual,Condition_Fair,Condition_Good,Condition_Poor,Condition_Very Good
132,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,False,False,False,...,False,True,False,False,False,False,False,False,False,True
317,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,False,False,False,...,False,False,False,False,True,False,False,True,False,False
234,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,False,False,False,...,True,False,False,False,False,True,False,True,False,False
312,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,False,False,False,...,False,False,False,True,True,False,False,True,False,False
232,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,False,False,False,...,False,False,False,False,False,True,False,False,False,True


In [29]:
print("Missing values in training data:", X_train.isnull().sum().sum())
print("Missing values in testing data:", X_test.isnull().sum().sum())

Missing values in training data: 0
Missing values in testing data: 0


In [30]:
X_train.dtypes

,0
Year,float64
Mileage_Km,float64
Engine_CC,float64
Power_BHP,float64
Previous_Owners,float64
...,...
Seller_Type_Individual,bool
Condition_Fair,bool
Condition_Good,bool
Condition_Poor,bool


In [31]:
processed_data = X_train.copy()
processed_data["Resale_Price_Lakh"] = y_train.values

processed_data.head()

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Car_ID_CAR0002,Car_ID_CAR0003,Car_ID_CAR0005,...,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Dealer,Seller_Type_Individual,Condition_Fair,Condition_Good,Condition_Poor,Condition_Very Good,Resale_Price_Lakh
132,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,False,False,False,...,True,False,False,False,False,False,False,False,True,4.26
317,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,False,False,False,...,False,False,False,True,False,False,True,False,False,5.30
234,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,False,False,False,...,False,False,False,False,True,False,True,False,False,1.23
312,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,False,False,False,...,False,False,True,True,False,False,True,False,False,7.09
232,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,False,False,False,...,False,False,False,False,True,False,False,False,True,2.69


In [32]:
processed_data.to_csv(
    "preprocessed_used_car_dataset.csv",
    index=False
)

print("Preprocessed dataset saved successfully!")

Preprocessed dataset saved successfully!


In [33]:
from google.colab import files

files.download("preprocessed_used_car_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>